# Employee Attrition Prediction - Data Ingestion

## Overview
This notebook loads and performs initial exploration of the HR Employee Attrition dataset. The data is loaded from a GitHub repository, converted into a Spark DataFrame, and explored using Databricks SQL to create a foundation for predictive modeling and employee turnover analysis.

## 1. Import Libraries

In [0]:
from pyspark.sql import functions as F
import pandas as pd

## 2. Load Dataset from GitHub

In [0]:
# Employee Attrition Prediction - Data Ingestion
# This notebook loads the HR Employee Attrition dataset into Databricks using PySpark.

# GitHub raw CSV URL
csv_url = "https://raw.githubusercontent.com/luc-dt/employee-attrition-prediction/main/data/HR_Employee_Attrition.csv"

# Load CSV into pandas
pdf = pd.read_csv(csv_url)

# Convert pandas DataFrame to Spark DataFrame
df = spark.createDataFrame(pdf)

# Create a temporary SQL view for analysis
df.createOrReplaceTempView("employee_attrition")

# Display first rows
display(df.limit(10))

# Show dataset size
print(f"Rows: {df.count()}")
print(f"Columns: {len(df.columns)}")

Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,EnvironmentSatisfaction,Gender,HourlyRate,JobInvolvement,JobLevel,JobRole,JobSatisfaction,MaritalStatus,MonthlyIncome,MonthlyRate,NumCompaniesWorked,Over18,OverTime,PercentSalaryHike,PerformanceRating,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,2,Female,94,3,2,Sales Executive,4,Single,5993,19479,8,Y,Yes,11,3,1,80,0,8,0,1,6,4,0,5
49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,3,Male,61,2,2,Research Scientist,2,Married,5130,24907,1,Y,No,23,4,4,80,1,10,3,3,10,7,1,7
37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,4,Male,92,2,1,Laboratory Technician,3,Single,2090,2396,6,Y,Yes,15,3,2,80,0,7,3,3,0,0,0,0
33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,4,Female,56,3,1,Research Scientist,3,Married,2909,23159,1,Y,Yes,11,3,3,80,0,8,3,3,8,7,3,0
27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,1,Male,40,3,1,Laboratory Technician,2,Married,3468,16632,9,Y,No,12,3,4,80,1,6,3,3,2,2,2,2
32,No,Travel_Frequently,1005,Research & Development,2,2,Life Sciences,1,8,4,Male,79,3,1,Laboratory Technician,4,Single,3068,11864,0,Y,No,13,3,3,80,0,8,2,2,7,7,3,6
59,No,Travel_Rarely,1324,Research & Development,3,3,Medical,1,10,3,Female,81,4,1,Laboratory Technician,1,Married,2670,9964,4,Y,Yes,20,4,1,80,3,12,3,2,1,0,0,0
30,No,Travel_Rarely,1358,Research & Development,24,1,Life Sciences,1,11,4,Male,67,3,1,Laboratory Technician,3,Divorced,2693,13335,1,Y,No,22,4,2,80,1,1,2,3,1,0,0,0
38,No,Travel_Frequently,216,Research & Development,23,3,Life Sciences,1,12,4,Male,44,2,3,Manufacturing Director,3,Single,9526,8787,0,Y,No,21,4,2,80,0,10,2,3,9,7,1,8
36,No,Travel_Rarely,1299,Research & Development,27,3,Medical,1,13,3,Male,94,3,2,Healthcare Representative,3,Married,5237,16577,6,Y,No,13,3,2,80,2,17,3,2,7,7,7,7


Rows: 1470
Columns: 35


## 3. Exploratory Data Analysis

### 3.1 Attrition Distribution Overview

In [0]:
%sql
SELECT
    Attrition,
    COUNT(*) AS employee_count,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS percentage
FROM employee_attrition
GROUP BY Attrition
ORDER BY Attrition;

Attrition,employee_count,percentage
No,1233,83.88
Yes,237,16.12


I first checked the target distribution in Databricks SQL and found that only about 16% of employees left. Because of this class imbalance, I evaluated recall/sensitivity instead of relying only on accuracy.

### 3.2 Attrition by OverTime Status

In [0]:
%sql
SELECT
    OverTime,
    COUNT(*) AS employee_count,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS attrition_count,
    ROUND(
        100.0 * SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS attrition_rate_pct
FROM employee_attrition
GROUP BY OverTime
ORDER BY attrition_rate_pct DESC;

OverTime,employee_count,attrition_count,attrition_rate_pct
Yes,416,127,30.53
No,1054,110,10.44


Employees who worked overtime had a 30.53% attrition rate, compared with only 10.44% for employees who did not work overtime. This suggests overtime is a major retention risk factor.

### 3.3 Early-Tenure Attrition by Department

In [0]:
%sql
SELECT
    Department,
    CASE
        WHEN YearsAtCompany < 1 THEN '<1 year'
        WHEN YearsAtCompany BETWEEN 1 AND 2 THEN '1-2 years'
        WHEN YearsAtCompany BETWEEN 3 AND 5 THEN '3-5 years'
        ELSE '6+ years'
    END AS tenure_band,
    COUNT(*) AS employee_count,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS attrition_count,
    ROUND(
        100.0 * SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS attrition_rate_pct
FROM employee_attrition
GROUP BY
    Department,
    CASE
        WHEN YearsAtCompany < 1 THEN '<1 year'
        WHEN YearsAtCompany BETWEEN 1 AND 2 THEN '1-2 years'
        WHEN YearsAtCompany BETWEEN 3 AND 5 THEN '3-5 years'
        ELSE '6+ years'
    END
ORDER BY attrition_rate_pct DESC, employee_count DESC;

Department,tenure_band,employee_count,attrition_count,attrition_rate_pct
Human Resources,1-2 years,13,6,46.15
Sales,<1 year,17,7,41.18
Sales,1-2 years,86,29,33.72
Research & Development,<1 year,27,9,33.33
Research & Development,1-2 years,199,51,25.63
Sales,3-5 years,117,24,20.51
Human Resources,3-5 years,23,4,17.39
Sales,6+ years,226,32,14.16
Research & Development,3-5 years,294,32,10.88
Research & Development,6+ years,441,41,9.30


Early-tenure employees, especially in Human Resources, Sales, and Research & Development, have higher attrition risk. HR should strengthen onboarding, manager check-ins, mentoring, and career development support during the first two years.

### 3.4 Business Travel and Commute Risk

In [0]:
%sql
SELECT
    BusinessTravel,
    CASE
        WHEN DistanceFromHome <= 5 THEN '0-5 miles'
        WHEN DistanceFromHome <= 15 THEN '6-15 miles'
        ELSE '16+ miles'
    END AS commute_band,
    COUNT(*) AS employee_count,
    SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS attrition_count,
    ROUND(AVG(JobSatisfaction), 2) AS avg_job_satisfaction,
    ROUND(
        100.0 * SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS attrition_rate_pct
FROM employee_attrition
GROUP BY
    BusinessTravel,
    CASE
        WHEN DistanceFromHome <= 5 THEN '0-5 miles'
        WHEN DistanceFromHome <= 15 THEN '6-15 miles'
        ELSE '16+ miles'
    END
HAVING COUNT(*) >= 10
ORDER BY attrition_rate_pct DESC;

BusinessTravel,commute_band,employee_count,attrition_count,avg_job_satisfaction,attrition_rate_pct
Travel_Frequently,16+ miles,66,20,2.94,30.30
Travel_Frequently,6-15 miles,86,21,2.73,24.42
Travel_Frequently,0-5 miles,125,28,2.75,22.40
Travel_Rarely,16+ miles,224,42,2.68,18.75
Non-Travel,16+ miles,39,6,2.69,15.38
Travel_Rarely,6-15 miles,375,57,2.66,15.20
Travel_Rarely,0-5 miles,444,57,2.75,12.84
Non-Travel,6-15 miles,48,4,2.9,8.33
Non-Travel,0-5 miles,63,2,2.78,3.17


Databricks visualization. Run in Databricks to view.

I used Databricks SQL to analyze attrition by business travel and commute distance. The highest attrition rate was among employees who traveled frequently and lived 16 or more miles from work, with a 30.30% attrition rate. This suggests that travel burden and commute distance may contribute to employee turnover.

## 4. Business Summary and Recommendations

This Databricks analysis identified several important attrition patterns:

1. **Class imbalance:** Only 16.12% of employees left the company, so model evaluation should focus on recall/sensitivity and not only accuracy.

2. **Overtime risk:** Employees who worked overtime had a 30.53% attrition rate, compared with 10.44% for employees who did not work overtime.

3. **Early-tenure risk:** Employees in their first two years showed higher attrition risk, especially in Human Resources, Sales, and Research & Development.

4. **Travel and commute risk:** Employees who traveled frequently and lived 16 or more miles from work had the highest travel/commute attrition rate at 30.30%.

### Recommended Actions

- Monitor overtime and workload distribution across departments.
- Improve onboarding and mentoring for employees in their first two years.
- Reduce unnecessary business travel where possible.
- Offer flexible work options for employees with long commutes.
- Track attrition trends over time to evaluate whether retention actions are working.